# RNN 实现古诗生成 —— 七言绝句

以 **"明月"** 为起始词，基于双层 LSTM 生成七言绝句。

**环境要求**
```bash
pip install torch matplotlib
```

**使用方法**：将本 Notebook 与四个 JSON 数据文件放在同一目录，逐单元格运行即可。

## 0. 安装依赖（首次运行时取消注释）

In [24]:
# !pip install torch matplotlib

## 1. 导入库 & 全局配置

In [25]:
import json, os, re, random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ─── 全局配置（按需修改） ───────────────────────────────────────────
CONFIG = {
    # 数据文件（与本 Notebook 同目录）
    "data_files": [
        "poet.song.40000.json",
        "poet.song.41000.json",
        "poet.song.42000.json",
        "poet.song.43000.json",
    ],
    "data_dir": "data_files",

    # 模型超参
    "embedding_dim": 256,
    "hidden_size":   512,
    "num_layers":    2,
    "dropout":       0.3,

    # 训练超参
    "batch_size":    64,
    "num_epochs":    30,
    "learning_rate": 1e-3,
    "lr_decay_step": 10,
    "lr_decay_gamma":0.5,
    "clip_grad":     5.0,
    "seed":          42,

    # 生成参数
    "start_words":   "明月",
    "temperature":   0.8,    # 采样温度，越低生成越保守

    # 输出路径
    "save_model":    "poem_lstm.pth",
    "loss_fig":      "training_loss.png",
}

# ─── 随机种子 & 设备 ─────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# 特殊 token
PAD, START, END, UNK = "<PAD>", "<START>", "<END>", "<UNK>"


Using device: cuda


## 2. 数据加载与预处理

**数据集格式说明**：JSON 中七言绝句存储为 **2 个 paragraph**，每个 16 字：
```
"XXXXXXX，XXXXXXX。"   ← 上下两句合为一联
```
过滤后展开为长度 32 的字符序列。

In [26]:
# 七言绝句每联格式：7汉字 + 逗号 + 7汉字 + 句号/！/？
_QIYAN_PATTERN = re.compile(
    r'^[\u4e00-\u9fff]{7}[，,][\u4e00-\u9fff]{7}[。！？]$'
)

def is_qiyan_jueju(paragraphs):
    """判断是否为七言绝句（2联，每联16字）"""
    if len(paragraphs) != 2:
        return False
    for p in paragraphs:
        ps = p.strip()
        if len(ps) != 16 or not _QIYAN_PATTERN.match(ps):
            return False
    return True


def load_sequences(data_dir, filenames):
    """读取 JSON，过滤七言绝句，每首展开为 32 字符序列（含标点）"""
    sequences = []
    for fname in filenames:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"[warning] 文件不存在，跳过: {fpath}")
            continue
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        for item in data:
            para = item.get("paragraphs", [])
            if is_qiyan_jueju(para):
                seq = "".join(p.strip() for p in para)  # 长度 32
                sequences.append(seq)

    print(f"过滤后七言绝句: {len(sequences)} 首")
    assert len(sequences) > 0, "未找到七言绝句！请检查 data_dir 配置。"
    return sequences


def build_vocab(sequences):
    """构建字符级词表"""
    chars = sorted(set("".join(sequences)))
    vocab = [PAD, START, END, UNK] + chars
    char2idx = {c: i for i, c in enumerate(vocab)}
    idx2char = {i: c for c, i in char2idx.items()}
    print(f"词表大小: {len(vocab)}")
    return char2idx, idx2char, vocab


# 执行加载
sequences        = load_sequences(CONFIG["data_dir"], CONFIG["data_files"])
char2idx, idx2char, vocab = build_vocab(sequences)

# 查看几条示例
print("\n示例序列：")
for s in sequences[:3]:
    print(" ", s)


过滤后七言绝句: 901 首
词表大小: 2827

示例序列：
  輕輕人問玄中旨，便吐肝腸說與他。木人暗皺雙眉處，石女多言爭奈何。
  圓缺曾伸問老翁，石龜銜子引清風。昨朝木馬潭中過，踏出金烏半夜紅。
  有無今古兩重關，正眼禪人過者難。欲通大道長安路，莫聽崑崙敘往還。


## 3. 构建 PyTorch Dataset

In [27]:
class PoemDataset(Dataset):
    """
    每个样本 (inp, tgt)，序列长度均为 32：
        inp = [START] + seq[:-1]   （前一字，i=0 时为 START）
        tgt = seq                   （当前字，作为预测目标）
    Teacher Forcing 训练范式。
    """
    def __init__(self, sequences, char2idx):
        start_id = char2idx[START]
        unk_id   = char2idx[UNK]
        self.data = []
        for seq in sequences:
            ids = [char2idx.get(c, unk_id) for c in seq]
            inp = torch.tensor([start_id] + ids[:-1], dtype=torch.long)
            tgt = torch.tensor(ids, dtype=torch.long)
            self.data.append((inp, tgt))

    def __len__(self):            return len(self.data)
    def __getitem__(self, i):     return self.data[i]


dataset = PoemDataset(sequences, char2idx)
print(f"训练样本数: {len(dataset)}")
inp_sample, tgt_sample = dataset[0]
print(f"inp shape: {inp_sample.shape},  tgt shape: {tgt_sample.shape}")


训练样本数: 901
inp shape: torch.Size([32]),  tgt shape: torch.Size([32])


## 4. 模型定义

结构：**Embedding(256) → 双层 LSTM(512) → Dropout(0.3) → Linear(vocab_size)**

In [28]:
class PoemLSTM(nn.Module):
    """
    字符级 LSTM 语言模型
    Embedding → 双层LSTM → Dropout → Linear(vocab_size)
    """
    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size  = emb_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, vocab_size)
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, hidden=None):
        """x: (B, L)  →  logits: (B, L, V), hidden"""
        emb    = self.dropout(self.embedding(x))   # (B, L, E)
        out, hidden = self.lstm(emb, hidden)        # (B, L, H)
        logits = self.fc(self.dropout(out))         # (B, L, V)
        return logits, hidden

    def init_hidden(self, batch, device):
        h = torch.zeros(self.lstm.num_layers, batch,
                        self.lstm.hidden_size, device=device)
        return (h, torch.zeros_like(h))


# 实例化模型
model = PoemLSTM(
    vocab_size  = len(vocab),
    emb_dim     = CONFIG["embedding_dim"],
    hidden_size = CONFIG["hidden_size"],
    num_layers  = CONFIG["num_layers"],
    dropout     = CONFIG["dropout"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数量: {n_params:,}")
print(model)


模型参数量: 5,852,171
PoemLSTM(
  (embedding): Embedding(2827, 256, padding_idx=0)
  (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=2827, bias=True)
)


## 5. 训练

In [29]:
def train_epoch(model, loader, optimizer, criterion, device, clip):
    """训练一个 epoch，返回平均 per-token loss"""
    model.train()
    total_loss, total_n = 0.0, 0
    for inp, tgt in loader:
        inp, tgt = inp.to(device), tgt.to(device)
        hidden = model.init_hidden(inp.size(0), device)

        logits, _ = model(inp, hidden)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        total_loss += loss.item() * tgt.numel()
        total_n    += tgt.numel()
    return total_loss / total_n


# ─── 初始化训练组件 ─────────────────────────────────────────────────
loader    = DataLoader(dataset, batch_size=CONFIG["batch_size"],
                       shuffle=True, num_workers=0)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=CONFIG["lr_decay_step"],
    gamma=CONFIG["lr_decay_gamma"])
criterion = nn.CrossEntropyLoss()

# ─── 训练循环 ───────────────────────────────────────────────────────
epoch_losses = []

for epoch in range(1, CONFIG["num_epochs"] + 1):
    loss = train_epoch(model, loader, optimizer, criterion,
                       DEVICE, CONFIG["clip_grad"])
    scheduler.step()
    epoch_losses.append(loss)

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Epoch [{epoch:02d}/{CONFIG['num_epochs']}]  "
          f"Loss: {loss:.4f}  LR: {lr_now:.2e}")

print("\n训练完成！")


Epoch [01/30]  Loss: 7.2121  LR: 1.00e-03
Epoch [02/30]  Loss: 6.6225  LR: 1.00e-03
Epoch [03/30]  Loss: 6.4959  LR: 1.00e-03
Epoch [04/30]  Loss: 6.2943  LR: 1.00e-03
Epoch [05/30]  Loss: 6.1923  LR: 1.00e-03
Epoch [06/30]  Loss: 6.1286  LR: 1.00e-03
Epoch [07/30]  Loss: 6.0393  LR: 1.00e-03
Epoch [08/30]  Loss: 5.9683  LR: 1.00e-03
Epoch [09/30]  Loss: 5.9176  LR: 1.00e-03
Epoch [10/30]  Loss: 5.8746  LR: 5.00e-04
Epoch [11/30]  Loss: 5.8282  LR: 5.00e-04
Epoch [12/30]  Loss: 5.7987  LR: 5.00e-04
Epoch [13/30]  Loss: 5.7746  LR: 5.00e-04
Epoch [14/30]  Loss: 5.7518  LR: 5.00e-04
Epoch [15/30]  Loss: 5.7301  LR: 5.00e-04
Epoch [16/30]  Loss: 5.7095  LR: 5.00e-04
Epoch [17/30]  Loss: 5.6830  LR: 5.00e-04
Epoch [18/30]  Loss: 5.6630  LR: 5.00e-04
Epoch [19/30]  Loss: 5.6415  LR: 5.00e-04
Epoch [20/30]  Loss: 5.6211  LR: 2.50e-04
Epoch [21/30]  Loss: 5.5959  LR: 2.50e-04
Epoch [22/30]  Loss: 5.5826  LR: 2.50e-04
Epoch [23/30]  Loss: 5.5698  LR: 2.50e-04
Epoch [24/30]  Loss: 5.5608  LR: 2

## 6. 保存模型

In [30]:
torch.save({
    "model_state_dict": model.state_dict(),
    "char2idx": char2idx,
    "idx2char":  idx2char,
    "vocab":     vocab,
    "config":    CONFIG,
}, CONFIG["save_model"])
print(f"模型已保存 → {CONFIG['save_model']}")


模型已保存 → poem_lstm.pth


## 7. 绘制 Loss 收敛曲线

In [31]:
fig, ax = plt.subplots(figsize=(9, 5))
epochs_x = list(range(1, len(epoch_losses) + 1))
ax.plot(epochs_x, epoch_losses, "b-o", markersize=4,
        linewidth=1.8, label="Train Loss")
ax.set_xlabel("Epoch", fontsize=13)
ax.set_ylabel("Loss",  fontsize=13)
ax.set_title("Training Loss Curve", fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_xticks(epochs_x)
fig.tight_layout()
fig.savefig(CONFIG["loss_fig"], dpi=150)
plt.show()
print(f"Loss 曲线 → {CONFIG['loss_fig']}")


Loss 曲线 → training_loss.png


C:\Windows\Temp\ipykernel_33888\318916420.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 8. 定义生成函数

**生成策略**：
1. 用 `[START] + start_words` 预热 LSTM hidden state
2. 从 `start_words` 最后一字开始自回归采样
3. 在标点位置（索引 7 / 15 / 23 / 31）**强制**插入正确标点
4. 其余位置屏蔽所有标点 token，再按温度 softmax 采样

In [32]:
# 七言绝句标点强制位置（0-based 序列索引）
PUNCT_MAP   = {7: "，", 15: "。", 23: "，", 31: "。"}
ALL_PUNCTS  = set("，。！？；、,.")


def generate(model, start_words, char2idx, idx2char, device,
             temperature=1.0, seq_len=32):
    """
    以 start_words 为前缀，自回归生成一首七言绝句（32 字含标点）。
    返回格式化的 4 行字符串。
    """
    model.eval()
    unk_id   = char2idx[UNK]
    start_id = char2idx[START]
    all_punct_ids = [char2idx[p] for p in ALL_PUNCTS if p in char2idx]

    generated = list(start_words)

    with torch.no_grad():
        # 预热 hidden state
        primer = [start_id] + [char2idx.get(c, unk_id) for c in start_words]
        inp    = torch.tensor([primer], dtype=torch.long, device=device)
        hidden = model.init_hidden(1, device)
        _, hidden = model(inp, hidden)

        # 自回归生成
        last_id = char2idx.get(start_words[-1], unk_id)
        inp = torch.tensor([[last_id]], dtype=torch.long, device=device)

        while len(generated) < seq_len:
            logits, hidden = model(inp, hidden)
            logit = logits[0, 0].clone() / temperature

            pos = len(generated)

            if pos in PUNCT_MAP:
                next_char = PUNCT_MAP[pos]
            else:
                logit[all_punct_ids] = -1e9
                probs = torch.softmax(logit, dim=-1)
                next_id = torch.multinomial(probs, 1).item()
                next_char = idx2char.get(next_id, UNK)

            generated.append(next_char)
            inp = torch.tensor(
                [[char2idx.get(next_char, unk_id)]],
                dtype=torch.long, device=device
            )

    s     = "".join(generated[:seq_len])
    lines = [s[i*8:(i+1)*8] for i in range(4)]
    return "\n".join(lines)

print("生成函数定义完毕。")


生成函数定义完毕。


## 9. 生成古诗

In [33]:
print("=" * 50)
print(f'以「{CONFIG["start_words"]}」为起始词生成七言绝句：')
print("=" * 50)

for i in range(5):
    poem = generate(
        model, CONFIG["start_words"],
        char2idx, idx2char, DEVICE,
        temperature=CONFIG["temperature"]
    )
    print(f"\n【第 {i+1} 首】\n{poem}")

print("\n" + "=" * 50)


以「明月」为起始词生成七言绝句：

【第 1 首】
明月朝色過空夕，
今將西圍松亦歸。
遽應高迎不藻走，
碧少還稱過鼓昏。

【第 2 首】
明月老得朝副去，
一趁幽聲往何幽。
爭樓明人靜抑我，
有衛曾俱寇無情。

【第 3 首】
明月前不香通施，
小公骸去理過層。
不將疏花半流約，
不觀白月松無人。

【第 4 首】
明月流煉青屏過，
只猶春繞終雲依。
昔人相長家業令，
掛在山綃不住時。

【第 5 首】
明月說得清平美，
夢相片雨不人歸。
不同物得世横意，
滿浩師樹自清蟠。



## 10. （可选）加载已保存模型重新生成

如果已经训练并保存了模型，可以跳过训练步骤，直接从这里加载生成。

In [35]:
checkpoint = torch.load(CONFIG["save_model"], map_location=DEVICE)

char2idx = checkpoint["char2idx"]
idx2char = checkpoint["idx2char"]
vocab    = checkpoint["vocab"]

model_loaded = PoemLSTM(
    vocab_size  = len(vocab),
    emb_dim     = CONFIG["embedding_dim"],
    hidden_size = CONFIG["hidden_size"],
    num_layers  = CONFIG["num_layers"],
    dropout     = CONFIG["dropout"],
).to(DEVICE)
model_loaded.load_state_dict(checkpoint["model_state_dict"])

poem = generate(model_loaded, "明月", char2idx, idx2char, DEVICE)
print(poem)


明月山眼書輪雪，
大語陰寥武晴柔。
只向紅來垂無舉，
貞知吹望憶迷頭。
